# ◆ Studio — Free Image + Video Generator

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Press ▶️ below and wait ~10 minutes.
3. Type a prompt, pick a style, press **Generate**.

Free Colab gives a few GPU hours per day. When it stops, press play again later.

In [ ]:
#@title ▶️ Press play. Wait ~10 minutes. The studio appears below.
import subprocess, os, gc, time, random
print("1/3 Installing ...")
subprocess.run("pip install -q diffusers transformers accelerate bitsandbytes sentencepiece gradio imageio imageio-ffmpeg protobuf", shell=True, check=True)

import torch, gradio as gr
from diffusers import StableDiffusionXLPipeline, LTXPipeline
from diffusers.utils import export_to_video
from transformers import T5EncoderModel, BitsAndBytesConfig

os.makedirs("/content/outputs", exist_ok=True)
state = {"name": None, "pipe": None}
history = []

def free():
    state["pipe"] = None; state["name"] = None
    gc.collect(); torch.cuda.empty_cache()

def get_image_pipe():
    if state["name"] != "image":
        free()
        p = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
        p.enable_vae_slicing()
        state["pipe"], state["name"] = p, "image"
    return state["pipe"]

def get_video_pipe():
    if state["name"] != "video":
        free()
        te = T5EncoderModel.from_pretrained("Lightricks/LTX-Video", subfolder="text_encoder",
                quantization_config=BitsAndBytesConfig(load_in_8bit=True), torch_dtype=torch.float16)
        p = LTXPipeline.from_pretrained("Lightricks/LTX-Video", text_encoder=te, torch_dtype=torch.float16)
        p.enable_model_cpu_offload()
        p.vae.enable_tiling()
        state["pipe"], state["name"] = p, "video"
    return state["pipe"]

STYLES = {
    "None": ("", ""),
    "Photorealistic": ("professional photograph, 85mm lens, natural lighting, highly detailed, sharp focus", "painting, drawing, cartoon, cgi"),
    "Cinematic": ("cinematic film still, dramatic lighting, shallow depth of field, color graded, 35mm", "cartoon, flat, low contrast"),
    "Anime": ("anime illustration, clean line art, vibrant colors, studio quality", "photo, realistic, 3d"),
    "3D Render": ("3d render, octane, unreal engine 5, global illumination, ultra detailed", "photo, sketch, flat"),
    "Watercolor": ("watercolor painting, soft edges, paper texture, delicate brush strokes", "photo, 3d, sharp"),
    "Digital Art": ("digital painting, concept art, trending on artstation, dramatic composition", "photo, blurry"),
}
SIZES = {"Square 1:1": (1024, 1024), "Portrait 3:4": (896, 1152), "Landscape 4:3": (1152, 896), "Widescreen 16:9": (1344, 768)}
BASE_NEG = "blurry, low quality, distorted, deformed, watermark, text, logo"

def build(prompt, style):
    s, n = STYLES[style]
    return (f"{prompt}, {s}" if s else prompt), (f"{BASE_NEG}, {n}" if n else BASE_NEG)

def make_image(prompt, style, size, count, steps, seed, progress=gr.Progress()):
    if not prompt.strip():
        raise gr.Error("Type a description first.")
    progress(0, desc="Loading model" if state["name"] != "image" else "Starting")
    p = get_image_pipe()
    pos, neg = build(prompt, style)
    w, h = SIZES[size]
    seed = int(seed) if int(seed) >= 0 else random.randint(0, 2**31)
    gen = torch.Generator("cuda").manual_seed(seed)
    def cb(pipe, i, t, kw):
        progress((i + 1) / steps, desc=f"Rendering step {i+1}/{steps}"); return kw
    imgs = p(prompt=pos, negative_prompt=neg, num_inference_steps=int(steps), guidance_scale=7,
             width=w, height=h, num_images_per_prompt=int(count), generator=gen,
             callback_on_step_end=cb).images
    for i, im in enumerate(imgs):
        path = f"/content/outputs/img_{int(time.time())}_{i}.png"; im.save(path); history.insert(0, path)
    return imgs, history[:24], f"Done · seed {seed} · {w}×{h}"

def make_video(prompt, style, seconds, steps, seed, progress=gr.Progress()):
    if not prompt.strip():
        raise gr.Error("Type a description first.")
    progress(0, desc="Loading model" if state["name"] != "video" else "Starting")
    p = get_video_pipe()
    pos, neg = build(prompt, style)
    seed = int(seed) if int(seed) >= 0 else random.randint(0, 2**31)
    gen = torch.Generator("cuda").manual_seed(seed)
    frames = int(seconds) * 24 + 1
    def cb(pipe, i, t, kw):
        progress((i + 1) / steps, desc=f"Rendering frame set {i+1}/{steps}"); return kw
    vid = p(prompt=pos, negative_prompt=neg, width=768, height=512, num_frames=frames,
            num_inference_steps=int(steps), guidance_scale=3, generator=gen,
            callback_on_step_end=cb).frames[0]
    path = f"/content/outputs/video_{int(time.time())}.mp4"
    export_to_video(vid, path, fps=24)
    return path, f"Done · seed {seed} · {seconds}s · 768×512"

CSS = """
.gradio-container {max-width: 1200px !important; margin: auto;}
#title {text-align:center; padding: 8px 0 0;}
#title h1 {font-size: 1.9rem; letter-spacing: -0.02em; margin-bottom: 2px;}
#title p {opacity: .65; margin-top: 0;}
#prompt textarea {font-size: 1.05rem;}
#go {height: 52px; font-size: 1.05rem;}
#status {opacity:.7; font-size:.9rem;}
footer {display:none !important;}
"""
theme = gr.themes.Base(primary_hue="indigo", neutral_hue="slate", font=gr.themes.GoogleFont("Inter")).set(
    body_background_fill="#0b0f19", body_background_fill_dark="#0b0f19",
    block_background_fill="#121826", block_background_fill_dark="#121826",
    block_border_width="1px", border_color_primary="#1f2937", border_color_primary_dark="#1f2937",
    input_background_fill="#0f1522", input_background_fill_dark="#0f1522",
    body_text_color="#e5e7eb", body_text_color_dark="#e5e7eb",
    block_label_text_color="#9ca3af", block_label_text_color_dark="#9ca3af",
    button_primary_background_fill="#4f46e5", button_primary_background_fill_hover="#4338ca",
    button_primary_text_color="white", block_radius="12px", button_large_radius="10px")

print("2/3 Building the studio ...")
with gr.Blocks(title="Studio", theme=theme, css=CSS) as app:
    gr.HTML('<div id="title"><h1>◆ Studio</h1><p>Free image &amp; video generation · running on your own GPU session</p></div>')
    with gr.Row():
        with gr.Column(scale=5):
            prompt = gr.Textbox(elem_id="prompt", label="Prompt", lines=3,
                placeholder="A lone lighthouse on a cliff at dusk, waves crashing, warm light in the window")
            with gr.Row():
                style = gr.Dropdown(list(STYLES), value="Photorealistic", label="Style")
                seed = gr.Number(value=-1, label="Seed (-1 = random)", precision=0)
        with gr.Column(scale=3):
            status = gr.Markdown("Ready.", elem_id="status")
    with gr.Tabs():
        with gr.Tab("Image"):
            with gr.Row():
                size = gr.Dropdown(list(SIZES), value="Square 1:1", label="Aspect ratio")
                count = gr.Slider(1, 4, value=1, step=1, label="Images")
                isteps = gr.Slider(15, 40, value=25, step=1, label="Quality (steps)")
            ibtn = gr.Button("Generate image", variant="primary", elem_id="go")
            igal = gr.Gallery(label="Result", columns=2, height=620, object_fit="contain", show_download_button=True)
        with gr.Tab("Video"):
            with gr.Row():
                secs = gr.Slider(2, 5, value=4, step=1, label="Length (seconds)")
                vsteps = gr.Slider(20, 40, value=30, step=1, label="Quality (steps)")
            vbtn = gr.Button("Generate video", variant="primary", elem_id="go")
            vout = gr.Video(label="Result", height=520)
        with gr.Tab("History"):
            hgal = gr.Gallery(label="This session", columns=4, height=620, object_fit="cover")
    ibtn.click(make_image, [prompt, style, size, count, isteps, seed], [igal, hgal, status])
    vbtn.click(make_video, [prompt, style, secs, vsteps, seed], [vout, status])
    prompt.submit(make_image, [prompt, style, size, count, isteps, seed], [igal, hgal, status])
    gr.Markdown("<div style='opacity:.5;font-size:.85rem;text-align:center'>First run of each tab downloads its model (a few minutes). Afterwards: images ≈30 s, videos 2–5 min. Files are also saved in /content/outputs.</div>")

print("3/3 Ready. Use the studio below (backup link printed underneath).")
app.queue().launch(share=True, inline=True, debug=False)
